# CoWork Enterprise Demo — Lab Setup
## Notebook 00 — Build the Foundation (Phase 0)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ Creates an **isolated lab role, database, and schemas** (`COWORK_ENTERPRISE_DEMO`) so nothing collides with existing objects on a shared account
2. 🛠️ Creates an **XS warehouse** and grants the demo admin role focused privileges
3. 🛠️ Enables **cross-region inference** (a safe no-op if already on)
4. 🛠️ Loads **self-contained sample data** for a fictional buy-side firm, *Nexus Capital* (clients, positions, trades, daily metrics, and unstructured research notes)
5. 🛠️ Validates every table loaded and the account is ready to build on

---

### Why This Is Phase 0:

🔹 In a real engagement, Phase 0 is to **confirm** the estate you already run — RBAC, network policies, MFA, masking, cross-region inference (see `PRODUCTION_HARDENING.md`). There is no estate to confirm in a demo, so this notebook instead **builds a clean sandbox** you can safely tear down later (Notebook 08).

**Run order:** 00 → 01 → 02 → 03 → 04 → 05 → 06 (dev→prod) → 07 (continuous improvement), then 08 (cleanup) last.

---

### Estimated Time: **10–15 minutes**

### Prerequisites:
- `ACCOUNTADMIN` (or a role that can create databases, warehouses, and roles)
- Enterprise Edition (for later notebooks: Cortex AI Guardrails, masking)

## Step 1: Create the Lab Role

🛠️ A dedicated role (`COWORK_ENTERPRISE_DEMO_ADMIN`) owns everything we build, so the whole demo can be granted and dropped as a unit — and nothing runs as `ACCOUNTADMIN` that doesn't have to.

In [ ]:
%%sql -r create_role
USE ROLE ACCOUNTADMIN;

-- Create lab role
CREATE ROLE IF NOT EXISTS COWORK_ENTERPRISE_DEMO_ADMIN
  COMMENT = 'Summit 2026 HOL - Nexus Capital lab role';

-- Grant role to current user and ACCOUNTADMIN
GRANT ROLE COWORK_ENTERPRISE_DEMO_ADMIN TO ROLE ACCOUNTADMIN;

SELECT 'COWORK_ENTERPRISE_DEMO_ADMIN role created and granted' AS STATUS;


## Step 2: Create the Database & Schemas

🛠️ Three schemas separate concerns: **ANALYTICS** (raw/gold tables), **SEMANTIC** (the governed semantic view), and **AGENTS** (the agent, search service, budgets). Ownership is transferred to the lab role so it can manage its own objects.

In [ ]:
%%sql -r create_db_schema
-- Database + 3 schemas (isolated demo namespace)
CREATE DATABASE IF NOT EXISTS COWORK_ENTERPRISE_DEMO COMMENT = 'CoWork Enterprise Demo - AI enterprise application';
CREATE SCHEMA IF NOT EXISTS COWORK_ENTERPRISE_DEMO.ANALYTICS COMMENT = 'Gold-layer analytical tables';
CREATE SCHEMA IF NOT EXISTS COWORK_ENTERPRISE_DEMO.SEMANTIC COMMENT = 'Semantic views and governed metrics';
CREATE SCHEMA IF NOT EXISTS COWORK_ENTERPRISE_DEMO.AGENTS COMMENT = 'Cortex Agents, Search services, budgets';
GRANT OWNERSHIP ON DATABASE COWORK_ENTERPRISE_DEMO TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN COPY CURRENT GRANTS;
GRANT OWNERSHIP ON ALL SCHEMAS IN DATABASE COWORK_ENTERPRISE_DEMO TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN COPY CURRENT GRANTS;
SELECT 'Database and schemas created' AS STATUS;


## Step 3: Create the Warehouse & Grant Privileges

🛠️ An XS warehouse runs the demo's SQL (agent LLM reasoning is serverless and billed separately). The grants give the lab role exactly what it needs to create semantic views, search services, agents, and tables — nothing more.

🔹 `SNOWFLAKE.CORTEX_USER` is the database role that entitles a role to call Cortex AI features.

In [ ]:
%%sql -r create_wh
-- Create warehouse
CREATE WAREHOUSE IF NOT EXISTS COWORK_ENTERPRISE_DEMO_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Summit 2026 HOL - Nexus Capital compute';

-- Grant usage to COWORK_ENTERPRISE_DEMO_ADMIN
GRANT USAGE ON WAREHOUSE COWORK_ENTERPRISE_DEMO_WH TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
GRANT OPERATE ON WAREHOUSE COWORK_ENTERPRISE_DEMO_WH TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;

SELECT 'COWORK_ENTERPRISE_DEMO_WH warehouse created (XS, auto-suspend 60s)' AS STATUS;


In [ ]:
%%sql -r grant_privileges
-- Focused privileges for the demo admin role
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
GRANT CREATE SEMANTIC VIEW ON SCHEMA COWORK_ENTERPRISE_DEMO.SEMANTIC TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
GRANT CREATE CORTEX SEARCH SERVICE ON SCHEMA COWORK_ENTERPRISE_DEMO.AGENTS TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
GRANT CREATE AGENT ON SCHEMA COWORK_ENTERPRISE_DEMO.AGENTS TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
GRANT CREATE TABLE ON SCHEMA COWORK_ENTERPRISE_DEMO.ANALYTICS TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
GRANT CREATE VIEW ON SCHEMA COWORK_ENTERPRISE_DEMO.ANALYTICS TO ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
SELECT 'Privileges granted' AS STATUS;


## Step 4: Enable Cross-Region Inference

🔹 Cortex may route a request to an LLM hosted in another region. This account-level setting permits that (required for some models and for evaluations). It is a **safe no-op** if already set.

In [ ]:
%%sql -r enable_cross_region
-- Enable cross-region inference (Global for best cost and model access)
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

SELECT 'Cross-region inference enabled (ANY_REGION)' AS STATUS;


## Step 5: Load the Sample Data

🛠️ Sets working context, then creates and loads the *Nexus Capital* dataset: **clients, portfolio positions, recent trades, daily metrics** (structured) plus **research notes** (unstructured, for Cortex Search). This is the data every later notebook builds on.

In [ ]:
%%sql -r use_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA ANALYTICS;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


In [ ]:
%%sql -r create_tables
-- CLIENTS table: Client master data
CREATE OR REPLACE TABLE COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS (
    CLIENT_ID           NUMBER AUTOINCREMENT PRIMARY KEY,
    CLIENT_NAME         VARCHAR(100) NOT NULL,
    ACCOUNT_TYPE        VARCHAR(20) NOT NULL,      -- INSTITUTIONAL, RETAIL, HNW
    REGION              VARCHAR(30) NOT NULL,
    AUM                 NUMBER(15,2),              -- Assets Under Management
    RISK_PROFILE        VARCHAR(20),               -- CONSERVATIVE, MODERATE, AGGRESSIVE
    ONBOARDED_DATE      DATE NOT NULL,
    RELATIONSHIP_MANAGER VARCHAR(50),
    EMAIL               VARCHAR(100),
    STATUS              VARCHAR(20) DEFAULT 'ACTIVE'
);

-- POSITIONS table: Current portfolio positions
CREATE OR REPLACE TABLE COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS (
    POSITION_ID         NUMBER AUTOINCREMENT PRIMARY KEY,
    CLIENT_ID           NUMBER NOT NULL,
    SYMBOL              VARCHAR(10) NOT NULL,
    QUANTITY            NUMBER NOT NULL,
    AVG_COST            NUMBER(12,4) NOT NULL,
    CURRENT_PRICE       NUMBER(12,4),
    MARKET_VALUE        NUMBER(15,2),
    UNREALIZED_PNL      NUMBER(15,2),
    SECTOR              VARCHAR(30),
    LAST_UPDATED        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- TRADES table: Recent trade executions
CREATE OR REPLACE TABLE COWORK_ENTERPRISE_DEMO.ANALYTICS.TRADES (
    TRADE_ID            NUMBER AUTOINCREMENT PRIMARY KEY,
    CLIENT_ID           NUMBER NOT NULL,
    SYMBOL              VARCHAR(10) NOT NULL,
    TRADE_TYPE          VARCHAR(4) NOT NULL,       -- BUY or SELL
    QUANTITY            NUMBER NOT NULL,
    PRICE               NUMBER(12,4) NOT NULL,
    TRADE_DATE          TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    SETTLEMENT_DATE     DATE,
    STATUS              VARCHAR(20) DEFAULT 'EXECUTED',
    EXCHANGE            VARCHAR(10) DEFAULT 'NYSE',
    NOTES               VARCHAR(500)
);

-- DAILY_METRICS table: Aggregated daily business metrics
CREATE OR REPLACE TABLE COWORK_ENTERPRISE_DEMO.ANALYTICS.DAILY_METRICS (
    METRIC_DATE         DATE NOT NULL,
    TOTAL_AUM           NUMBER(18,2),
    TOTAL_CLIENTS       NUMBER,
    ACTIVE_CLIENTS      NUMBER,
    TOTAL_TRADES        NUMBER,
    TRADE_VOLUME_USD    NUMBER(18,2),
    NET_FLOWS           NUMBER(15,2),
    TOTAL_REVENUE       NUMBER(15,2),
    AVG_PORTFOLIO_RISK  NUMBER(5,3),
    TOP_SECTOR          VARCHAR(30)
);

SELECT 'Analytics tables created' AS STATUS;


In [ ]:
%%sql -r load_clients
-- Load client data
INSERT INTO COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS (CLIENT_NAME, ACCOUNT_TYPE, REGION, AUM, RISK_PROFILE, ONBOARDED_DATE, RELATIONSHIP_MANAGER, EMAIL, STATUS)
VALUES
('Meridian Pension Fund', 'INSTITUTIONAL', 'North America', 2500000000.00, 'MODERATE', '2019-03-15', 'Sarah Chen', 'contact@meridianpf.com', 'ACTIVE'),
('Atlas Sovereign Wealth', 'INSTITUTIONAL', 'EMEA', 8700000000.00, 'CONSERVATIVE', '2017-08-22', 'James Morrison', 'invest@atlasswf.com', 'ACTIVE'),
('Velocity Capital Partners', 'INSTITUTIONAL', 'North America', 1200000000.00, 'AGGRESSIVE', '2020-01-10', 'Sarah Chen', 'ops@velocitycap.com', 'ACTIVE'),
('Sakura Asset Management', 'INSTITUTIONAL', 'APJ', 3100000000.00, 'MODERATE', '2018-11-05', 'Kenji Tanaka', 'info@sakura-am.jp', 'ACTIVE'),
('Nordic Growth Fund', 'INSTITUTIONAL', 'EMEA', 950000000.00, 'AGGRESSIVE', '2021-06-18', 'Erik Lindqvist', 'invest@nordicgf.se', 'ACTIVE'),
('Wellington Family Office', 'HNW', 'North America', 450000000.00, 'MODERATE', '2016-02-28', 'Sarah Chen', 'pw@wellingtonfamily.com', 'ACTIVE'),
('Horizon Endowment', 'INSTITUTIONAL', 'North America', 1800000000.00, 'CONSERVATIVE', '2015-09-12', 'Michael Brooks', 'endowment@horizonedu.org', 'ACTIVE'),
('Pacific Rim Ventures', 'INSTITUTIONAL', 'APJ', 2200000000.00, 'AGGRESSIVE', '2019-07-30', 'Kenji Tanaka', 'invest@pacificrimv.sg', 'ACTIVE'),
('Blackstone Ridge Capital', 'HNW', 'North America', 680000000.00, 'AGGRESSIVE', '2022-03-01', 'Michael Brooks', 'admin@blackstoneridge.com', 'ACTIVE'),
('Emirates Diversified Holdings', 'INSTITUTIONAL', 'EMEA', 5400000000.00, 'MODERATE', '2018-01-20', 'James Morrison', 'invest@emiratesdh.ae', 'ACTIVE'),
('Redwood Retirement Systems', 'INSTITUTIONAL', 'North America', 4100000000.00, 'CONSERVATIVE', '2014-05-15', 'Sarah Chen', 'ops@redwoodretire.com', 'ACTIVE'),
('Chen Wei Holdings', 'HNW', 'APJ', 320000000.00, 'MODERATE', '2021-11-22', 'Kenji Tanaka', 'office@chenwei.hk', 'ACTIVE');

SELECT 'Clients loaded: ' || COUNT(*) || ' rows' AS STATUS FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS;


In [ ]:
%%sql -r load_positions
-- Load portfolio positions
INSERT INTO COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS (CLIENT_ID, SYMBOL, QUANTITY, AVG_COST, CURRENT_PRICE, MARKET_VALUE, UNREALIZED_PNL, SECTOR)
VALUES
(1, 'AAPL', 50000, 145.2300, 198.5000, 9925000.00, 2663500.00, 'Technology'),
(1, 'MSFT', 35000, 280.5000, 425.3000, 14885500.00, 5068000.00, 'Technology'),
(1, 'JNJ', 40000, 155.8000, 162.4000, 6496000.00, 264000.00, 'Healthcare'),
(2, 'BRK.B', 25000, 310.0000, 465.2000, 11630000.00, 3880000.00, 'Financials'),
(2, 'PG', 60000, 138.5000, 168.9000, 10134000.00, 1824000.00, 'Consumer Staples'),
(2, 'XOM', 45000, 85.6000, 112.3000, 5053500.00, 1201500.00, 'Energy'),
(3, 'NVDA', 80000, 42.5000, 135.8000, 10864000.00, 7464000.00, 'Technology'),
(3, 'TSLA', 30000, 185.0000, 248.5000, 7455000.00, 1905000.00, 'Consumer Discretionary'),
(3, 'AMD', 55000, 95.3000, 168.7000, 9278500.00, 4037000.00, 'Technology'),
(4, 'SONY', 90000, 78.5000, 95.2000, 8568000.00, 1503000.00, 'Technology'),
(4, 'TM', 40000, 155.0000, 182.4000, 7296000.00, 1096000.00, 'Consumer Discretionary'),
(5, 'ASML', 12000, 580.0000, 890.5000, 10686000.00, 3726000.00, 'Technology'),
(5, 'SPOT', 25000, 145.0000, 328.6000, 8215000.00, 4590000.00, 'Technology'),
(6, 'AAPL', 20000, 130.0000, 198.5000, 3970000.00, 1370000.00, 'Technology'),
(6, 'V', 15000, 220.0000, 295.8000, 4437000.00, 1137000.00, 'Financials'),
(7, 'VTI', 100000, 195.0000, 268.4000, 26840000.00, 7340000.00, 'Broad Market'),
(7, 'BND', 80000, 72.5000, 69.8000, 5584000.00, -216000.00, 'Fixed Income'),
(8, 'BABA', 120000, 88.0000, 125.3000, 15036000.00, 4476000.00, 'Technology'),
(8, 'SE', 65000, 52.0000, 98.5000, 6402500.00, 3022500.00, 'Technology'),
(9, 'COIN', 35000, 65.0000, 225.4000, 7889000.00, 5614000.00, 'Financials'),
(9, 'MSTR', 18000, 320.0000, 1650.0000, 29700000.00, 23940000.00, 'Technology'),
(10, 'ADNOC', 200000, 3.2000, 4.1000, 820000.00, 180000.00, 'Energy'),
(10, 'SAP', 30000, 145.0000, 235.6000, 7068000.00, 2718000.00, 'Technology'),
(11, 'VTI', 150000, 180.0000, 268.4000, 40260000.00, 13260000.00, 'Broad Market'),
(11, 'SCHD', 200000, 72.0000, 82.5000, 16500000.00, 2100000.00, 'Dividends'),
(12, 'TCEHY', 50000, 42.0000, 58.3000, 2915000.00, 815000.00, 'Technology'),
(12, '9988.HK', 80000, 72.5000, 108.2000, 8656000.00, 2856000.00, 'Technology');

SELECT 'Positions loaded: ' || COUNT(*) || ' rows' AS STATUS FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS;


In [ ]:
%%sql -r load_trades
-- Load recent trades
INSERT INTO COWORK_ENTERPRISE_DEMO.ANALYTICS.TRADES (CLIENT_ID, SYMBOL, TRADE_TYPE, QUANTITY, PRICE, TRADE_DATE, SETTLEMENT_DATE, STATUS, EXCHANGE, NOTES)
VALUES
(1, 'AAPL', 'BUY', 5000, 197.80, DATEADD('hour', -2, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Adding to core tech position'),
(1, 'MSFT', 'BUY', 2000, 424.50, DATEADD('hour', -3, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Pre-earnings accumulation'),
(3, 'NVDA', 'SELL', 10000, 136.20, DATEADD('hour', -1, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Profit taking after run-up'),
(3, 'TSLA', 'BUY', 8000, 247.90, DATEADD('hour', -4, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Momentum entry'),
(2, 'XOM', 'SELL', 5000, 111.80, DATEADD('hour', -5, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Reducing energy exposure'),
(5, 'ASML', 'BUY', 2000, 892.00, DATEADD('minute', -30, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Semiconductor thesis'),
(4, 'SONY', 'BUY', 15000, 94.80, DATEADD('hour', -6, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Gaming sector exposure'),
(9, 'COIN', 'BUY', 10000, 224.50, DATEADD('minute', -45, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Crypto proxy play'),
(6, 'V', 'BUY', 3000, 294.20, DATEADD('hour', -7, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Payments thesis'),
(8, 'SE', 'SELL', 20000, 99.10, DATEADD('hour', -2, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Taking profits on SE Asia tech'),
(7, 'BND', 'BUY', 25000, 69.90, DATEADD('hour', -8, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Rebalancing fixed income allocation'),
(10, 'SAP', 'BUY', 5000, 236.10, DATEADD('hour', -3, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Enterprise software conviction'),
(11, 'SCHD', 'BUY', 30000, 82.30, DATEADD('day', -1, CURRENT_TIMESTAMP()), DATEADD('day', 1, CURRENT_DATE()), 'SETTLED', 'NYSE', 'Dividend reinvestment'),
(12, 'TCEHY', 'BUY', 10000, 57.80, DATEADD('day', -1, CURRENT_TIMESTAMP()), DATEADD('day', 1, CURRENT_DATE()), 'SETTLED', 'OTC', 'China tech rebound thesis'),
(1, 'JNJ', 'SELL', 10000, 163.00, DATEADD('day', -1, CURRENT_TIMESTAMP()), DATEADD('day', 1, CURRENT_DATE()), 'SETTLED', 'NYSE', 'Trimming healthcare overweight'),
(3, 'AMD', 'BUY', 12000, 167.50, DATEADD('minute', -90, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'AI infrastructure play'),
(5, 'SPOT', 'SELL', 5000, 330.00, DATEADD('hour', -4, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Locking in gains'),
(2, 'PG', 'BUY', 10000, 169.20, DATEADD('hour', -5, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Defensive positioning'),
(4, 'TM', 'SELL', 8000, 183.00, DATEADD('hour', -6, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NYSE', 'Rotating out of auto'),
(9, 'MSTR', 'BUY', 3000, 1645.00, DATEADD('minute', -20, CURRENT_TIMESTAMP()), DATEADD('day', 2, CURRENT_DATE()), 'EXECUTED', 'NASDAQ', 'Bitcoin proxy - high conviction');

SELECT 'Trades loaded: ' || COUNT(*) || ' rows' AS STATUS FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.TRADES;


In [ ]:
%%sql -r load_metrics
-- Load daily metrics (last 30 days of aggregated business metrics)
INSERT INTO COWORK_ENTERPRISE_DEMO.ANALYTICS.DAILY_METRICS (METRIC_DATE, TOTAL_AUM, TOTAL_CLIENTS, ACTIVE_CLIENTS, TOTAL_TRADES, TRADE_VOLUME_USD, NET_FLOWS, TOTAL_REVENUE, AVG_PORTFOLIO_RISK, TOP_SECTOR)
SELECT
    DATEADD('day', -seq4(), CURRENT_DATE()) AS METRIC_DATE,
    31400000000.00 + (RANDOM() % 500000000) AS TOTAL_AUM,
    12 AS TOTAL_CLIENTS,
    CASE WHEN seq4() % 7 = 0 THEN 10 ELSE 12 END AS ACTIVE_CLIENTS,
    15 + (ABS(RANDOM()) % 20) AS TOTAL_TRADES,
    25000000.00 + (ABS(RANDOM()) % 50000000) AS TRADE_VOLUME_USD,
    -5000000.00 + (ABS(RANDOM()) % 10000000) AS NET_FLOWS,
    850000.00 + (ABS(RANDOM()) % 300000) AS TOTAL_REVENUE,
    0.45 + (ABS(RANDOM()) % 20) / 100.0 AS AVG_PORTFOLIO_RISK,
    CASE (ABS(RANDOM()) % 4)
        WHEN 0 THEN 'Technology'
        WHEN 1 THEN 'Financials'
        WHEN 2 THEN 'Energy'
        ELSE 'Healthcare'
    END AS TOP_SECTOR
FROM TABLE(GENERATOR(ROWCOUNT => 30));

SELECT 'Daily metrics loaded: ' || COUNT(*) || ' rows' AS STATUS FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.DAILY_METRICS;


In [ ]:
%%sql -r load_research
-- Research notes and market commentary (for Cortex Search)
CREATE OR REPLACE TABLE COWORK_ENTERPRISE_DEMO.ANALYTICS.RESEARCH_NOTES (
    NOTE_ID         NUMBER AUTOINCREMENT PRIMARY KEY,
    TITLE           VARCHAR(200) NOT NULL,
    CONTENT         VARCHAR(5000) NOT NULL,
    AUTHOR          VARCHAR(50),
    CATEGORY        VARCHAR(30),
    PUBLISHED_DATE  DATE,
    REGION          VARCHAR(30),
    SYMBOLS_MENTIONED VARCHAR(200)
);

INSERT INTO COWORK_ENTERPRISE_DEMO.ANALYTICS.RESEARCH_NOTES (TITLE, CONTENT, AUTHOR, CATEGORY, PUBLISHED_DATE, REGION, SYMBOLS_MENTIONED)
VALUES
('Q2 Technology Sector Outlook', 'The technology sector continues to benefit from AI infrastructure buildout. NVIDIA remains the primary beneficiary of data center GPU demand, with hyperscaler capex showing no signs of deceleration. We maintain overweight positions in semiconductor names (NVDA, AMD, ASML) and see further upside in AI-adjacent software companies. Risk: elevated valuations leave limited margin of safety if earnings disappoint.', 'Sarah Chen', 'Market Commentary', CURRENT_DATE() - 5, 'North America', 'NVDA,AMD,ASML'),
('Emerging Markets Risk Assessment', 'Geopolitical tensions in the South China Sea and ongoing US-China trade restrictions create headwinds for APJ equity exposure. However, Japanese equities (SONY, TM) benefit from yen weakness and corporate governance reforms. We recommend selective positioning in Japan while maintaining caution on Greater China names. Sakura Asset Management has expressed interest in increasing Japan allocation.', 'Kenji Tanaka', 'Risk Assessment', CURRENT_DATE() - 3, 'APJ', 'SONY,TM,TCEHY,BABA'),
('Fixed Income Rebalancing Strategy', 'With rate cuts expected in H2, we recommend gradually extending duration in fixed income portfolios. Horizon Endowment and Redwood Retirement Systems should consider shifting from short-duration BND positions to intermediate-term corporates. The yield curve normalization thesis supports this move. Current BND position shows unrealized loss of $216K — acceptable given the strategic rationale.', 'Michael Brooks', 'Strategy', CURRENT_DATE() - 7, 'North America', 'BND,SCHD'),
('EMEA Energy Transition Update', 'European energy majors are accelerating renewable investments. Emirates Diversified Holdings should maintain XOM exposure for dividend yield but consider adding renewable energy ETFs for long-term positioning. SAP enterprise software adoption continues to accelerate across EMEA — strong conviction on the digital transformation thesis.', 'James Morrison', 'Sector Update', CURRENT_DATE() - 2, 'EMEA', 'XOM,SAP'),
('Crypto and Digital Assets Thesis', 'Bitcoin proxy plays (COIN, MSTR) have outperformed direct crypto holdings on a risk-adjusted basis. Blackstone Ridge Capital has generated exceptional returns on MSTR position (+$23.9M unrealized). We recommend maintaining but not increasing allocation — concentration risk is elevated. Consider taking partial profits if MSTR exceeds $1,800.', 'Michael Brooks', 'Thematic Research', CURRENT_DATE() - 1, 'North America', 'COIN,MSTR'),
('Client Portfolio Compliance Review - May 2026', 'Monthly compliance check complete. All portfolios within risk mandate boundaries. One flagged item: Velocity Capital Partners TSLA position approaching 15% single-name concentration limit after recent BUY order (8,000 shares at $247.90). Recommend monitoring and potential trim if position exceeds threshold. No other policy breaches detected across 12 active accounts.', 'Compliance Team', 'Compliance', CURRENT_DATE(), 'North America', 'TSLA'),
('Nordic Growth Fund - Quarterly Review', 'Nordic Growth Fund continues to outperform benchmark with aggressive tech allocation (ASML, SPOT). ASML position has generated $3.7M unrealized gains on semiconductor thesis. Spotify gains reflect subscriber growth exceeding expectations. Fund manager Erik Lindqvist has requested exploration of AI infrastructure names — recommend presenting NVDA and AMD analysis at next quarterly meeting.', 'Erik Lindqvist', 'Client Review', CURRENT_DATE() - 10, 'EMEA', 'ASML,SPOT,NVDA,AMD'),
('Pacific Rim Ventures - Rotation Strategy', 'Client has requested rotation out of SE Asia tech (SE position) into broader AI plays. The $3M+ unrealized gain on SE provides tax-loss harvesting opportunity if paired with appropriate offset. Recommend phased exit over 2 weeks to minimize market impact. Replacement candidates: BABA (already held), or new positions in Korean AI chip names.', 'Kenji Tanaka', 'Trading Strategy', CURRENT_DATE() - 4, 'APJ', 'SE,BABA');

SELECT 'Research notes loaded: ' || COUNT(*) || ' rows' AS STATUS FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.RESEARCH_NOTES;


## Step 6: Validate

📌 Confirm every table loaded and cross-region inference is set before moving on — later notebooks assume this data exists.

In [ ]:
%%sql -r validate_setup
-- Validate all tables and row counts
SELECT 'CLIENTS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
UNION ALL
SELECT 'POSITIONS', COUNT(*) FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS
UNION ALL
SELECT 'TRADES', COUNT(*) FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.TRADES
UNION ALL
SELECT 'DAILY_METRICS', COUNT(*) FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.DAILY_METRICS
UNION ALL
SELECT 'RESEARCH_NOTES', COUNT(*) FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.RESEARCH_NOTES
ORDER BY TABLE_NAME;


In [ ]:
%%sql -r sanity_check
-- Quick sanity check: Top 5 clients by AUM
SELECT CLIENT_NAME, ACCOUNT_TYPE, REGION, AUM, RISK_PROFILE
FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
ORDER BY AUM DESC
LIMIT 5;


In [ ]:
%%sql -r verify_cross_region
-- Verify cross-region inference is enabled
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;


## ✅ Notebook 00 Complete

### What You Just Built:
- Lab role `COWORK_ENTERPRISE_DEMO_ADMIN` + database `COWORK_ENTERPRISE_DEMO` with ANALYTICS / SEMANTIC / AGENTS schemas
- XS warehouse `COWORK_ENTERPRISE_DEMO_WH` and focused grants
- Cross-region inference enabled
- *Nexus Capital* sample data: clients, positions, trades, daily metrics, and research notes

---

### Key Points:
- Everything lives under one database + role, so the demo is isolated on a shared account and trivially removable (Notebook 08).
- The data layer is the foundation — semantic-view and agent quality depend on it.

---

### Next:

Continue to **Notebook 01 — Build the Context Layer**: define a governed semantic view over this data and a Cortex Search service over the research notes.